In [0]:
import json

credentials_info = json.loads(
    dbutils.secrets.get(
        scope="openbanking",
        key="gmail-token",
    )
)

required = ("refresh_token", "client_id", "client_secret", "token_uri")
missing = [field for field in required if not credentials_info.get(field)]

if missing:
    raise ValueError(f"Faltan campos: {', '.join(missing)}")

print("Credenciales de Gmail recuperadas correctamente.")

In [0]:
%pip install google-api-python-client google-auth

In [0]:
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build

credentials = Credentials.from_authorized_user_info(
    credentials_info,
    scopes=["https://www.googleapis.com/auth/gmail.readonly"],
)

# Comprobar que el refresh token permite obtener acceso nuevo.
credentials.refresh(Request())

gmail = build(
    "gmail",
    "v1",
    credentials=credentials,
    cache_discovery=False,
)

result = gmail.users().messages().list(
    userId="me",
    q=(
        'from:postamail@laboralkutxa.com '
        'subject:"POSTAMAIL - LABORAL KUTXA" '
        'has:attachment filename:pdf'
    ),
    maxResults=10,
).execute()

print("Conexión a Gmail correcta.")
print("Correos devueltos en esta página:", len(result.get("messages", [])))
print("Hay más páginas:", bool(result.get("nextPageToken")))

In [0]:
def walk_parts(part):
    yield part
    for child in part.get("parts", []):
        yield from walk_parts(child)


matched_pdfs = 0

for item in result.get("messages", []):
    message = gmail.users().messages().get(
        userId="me",
        id=item["id"],
        format="full",
    ).execute()

    for part in walk_parts(message.get("payload", {})):
        filename = part.get("filename", "")

        if (
            filename.lower().endswith(".pdf")
            and "EXTRACTO_TARJETA_VISA_CLASICA_GO" in filename.upper()
        ):
            matched_pdfs += 1

print("Extractos VISA encontrados:", matched_pdfs)

In [0]:
%pip install azure-storage-blob

In [0]:
import base64
import hashlib
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

from azure.core.exceptions import ResourceExistsError

source_path = str(Path.cwd().parent / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)

from openbanking.storage import Storage

storage = Storage(
    account_name="graxianodatalake",
    account_key=dbutils.secrets.get("azure-storage", "graxiano"),
    container_name="raw",
)

prefix = "personal/finanzas/openbanking/gmail/laboral_kutxa"
uploaded = 0
existing = 0

for item in result.get("messages", []):
    message = gmail.users().messages().get(
        userId="me",
        id=item["id"],
        format="full",
    ).execute()

    for part in walk_parts(message.get("payload", {})):
        filename = part.get("filename", "")

        if not (
            filename.lower().endswith(".pdf")
            and "EXTRACTO_TARJETA_VISA_CLASICA_GO" in filename.upper()
        ):
            continue

        body = part.get("body", {})

        if body.get("attachmentId"):
            body = gmail.users().messages().attachments().get(
                userId="me",
                messageId=item["id"],
                id=body["attachmentId"],
            ).execute()

        encoded = body.get("data", "")
        if not encoded:
            raise ValueError("El adjunto está vacío.")

        content = base64.urlsafe_b64decode(
            encoded + "=" * (-len(encoded) % 4)
        )

        if not content.startswith(b"%PDF-"):
            raise ValueError("El adjunto no tiene una cabecera PDF válida.")

        digest = hashlib.sha256(content).hexdigest()
        pdf_path = f"{prefix}/{digest}.pdf"

        metadata = {
            "message_id": item["id"],
            "message_internal_date_ms": message.get("internalDate"),
            "original_filename": filename,
            "sha256": digest,
            "size_bytes": len(content),
            "extracted_at": datetime.now(timezone.utc).isoformat(),
        }

        try:
            storage.upload(pdf_path, content)
            uploaded += 1
        except ResourceExistsError:
            if storage.download(pdf_path) != content:
                raise RuntimeError("El PDF existente no coincide.")
            existing += 1

        # También completa los metadatos si una ejecución anterior
        # guardó el PDF pero falló antes de guardar su JSON.
        try:
            storage.upload(
                f"{prefix}/{digest}.json",
                json.dumps(
                    metadata, ensure_ascii=False, indent=2
                ).encode("utf-8"),
            )
        except ResourceExistsError:
            pass

print("PDF nuevos guardados:", uploaded)
print("PDF ya existentes:", existing)

## Call SRC/OPENBANKING/gmail_client.py

In [0]:
from openbanking.gmail_client import GmailClient

gmail_client = GmailClient(credentials_info)

count = sum(1 for _ in gmail_client.iter_visa_pdfs())

print("Extractos VISA recuperados por el módulo:", count)

## Call SRC/OPENBANKING/ingest_gmail.py

In [0]:
from openbanking.ingest_gmail import ingest_gmail

summary = ingest_gmail(
    gmail_client=gmail_client,
    storage=storage,
    prefix="personal/finanzas/openbanking/gmail/laboral_kutxa",
)

print("PDF nuevos guardados:", summary["uploaded"])
print("PDF ya existentes:", summary["existing"])

In [0]:
from importlib.metadata import version

for package in (
    "azure-storage-blob",
    "google-api-python-client",
    "google-auth",
):
    print(f"{package}=={version(package)}")